# Shopify API spike

Exploratory notebook for connecting to the Shopify Admin API and generating an
OAuth authorization URL, using the Python `shopify` library.

**Credentials are read from environment variables** - never hard-code them:

```bash
export SHOPIFY_API_KEY=...            # app client id / API key
export SHOPIFY_API_SECRET=...         # OAuth shared secret (shpss_...)
export SHOPIFY_ADMIN_TOKEN=...        # Admin API access token (shpat_...)
export SHOPIFY_SHOP_DOMAIN=your-store.myshopify.com
```

Credential types matter for the Admin API:
- `shpat_...` - custom-app Admin API access token (use this for API calls)
- `shpss_...` - OAuth shared secret (used to validate the OAuth callback, **not** an API token)

In [ ]:
import os
import binascii

import shopify

API_KEY = os.environ["SHOPIFY_API_KEY"]
API_SECRET = os.environ["SHOPIFY_API_SECRET"]
SHOP_DOMAIN = os.environ.get("SHOPIFY_SHOP_DOMAIN", "your-store.myshopify.com")
API_VERSION = "2024-07"

## 1. Build an OAuth authorization URL

Redirect the merchant to this URL to grant access; Shopify redirects back to
`redirect_uri` with a `code` you exchange for an access token.

In [ ]:
shopify.Session.setup(api_key=API_KEY, secret=API_SECRET)

state = binascii.b2a_hex(os.urandom(15)).decode("utf-8")
redirect_uri = "https://your-app.example.com/auth/shopify/callback"
scopes = ["read_products", "read_orders"]

session = shopify.Session(SHOP_DOMAIN, API_VERSION)
auth_url = session.create_permission_url(
    redirect_uri=redirect_uri, scope=scopes, state=state
)
print(auth_url)

## 2. Exchange the callback code for a token

After the merchant approves, Shopify redirects to `redirect_uri` with query
params. Pass those params to `request_token`; it validates the HMAC and returns
the access token, which you should store securely for future requests.

In [ ]:
# callback_params = dict(request.GET)  # from your web framework
# session = shopify.Session(SHOP_DOMAIN, API_VERSION)
# access_token = session.request_token(callback_params)
# store_token_securely(SHOP_DOMAIN, access_token)

## 3. Smoke test against the Admin API

Uses an existing Admin API access token (`shpat_...`) from the environment.

In [ ]:
access_token = os.environ["SHOPIFY_ADMIN_TOKEN"]

session = shopify.Session(SHOP_DOMAIN, API_VERSION, access_token)
shopify.ShopifyResource.activate_session(session)

try:
    result = shopify.GraphQL().execute(
        "{ shop { name email myshopifyDomain plan { displayName } } }"
    )
    print("Connected. Shop info:")
    print(result)
except Exception as e:
    print(f"Connection failed: {type(e).__name__}: {e}")
    print(
        "If this is a 401, the token isn't a valid Admin API access token. "
        "Create a custom app in the Shopify admin and use its shpat_... token."
    )
finally:
    shopify.ShopifyResource.clear_session()